## WinTSR quickstart — interpret a time series model in 60 seconds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/quickstart.ipynb)

**Windowed Temporal Saliency Rescaling (WinTSR)** tells you *which time steps and which
features* a trained time series model actually used. It works on any PyTorch model that
takes `(batch, seq_len, n_features)` — no need to adopt any training framework.

This notebook needs **no dataset download**. We generate a series where we know the
ground-truth answer, train a small GRU on it, and check that WinTSR recovers it.

Paper: [arXiv:2412.04532](https://arxiv.org/abs/2412.04532)

In [ ]:
%pip install -q tslens matplotlib scikit-learn
# From source instead:
# %pip install -q "git+https://github.com/khairulislam/tslens.git"

## 1. A dataset where we know the right answer

We build 5 autocorrelated (AR(1)) input channels of length 50. The target depends on
**one feature (0), over one window (time steps 20–25)**. Everything else is noise.

That known window is our ground truth: a good interpretability method should light up
there and nowhere else.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)

SEQ_LEN, N_FEATURES = 50, 5
SIGNAL_FEATURE, SIGNAL_START, SIGNAL_END = 0, 20, 26
RHO = 0.9  # autocorrelation: real time series are not i.i.d. noise


def ar_series(n, rho=RHO):
    eps = torch.randn(n, SEQ_LEN, N_FEATURES)
    x = torch.empty_like(eps)
    x[:, 0] = eps[:, 0]
    for t in range(1, SEQ_LEN):
        x[:, t] = rho * x[:, t - 1] + (1 - rho ** 2) ** 0.5 * eps[:, t]
    return x


def make(n):
    x = ar_series(n)
    y = x[:, SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE].sum(dim=1, keepdim=True)
    return x, y


x_train, y_train = make(4000)
x_test, y_test = make(256)

ground_truth = torch.zeros(SEQ_LEN, N_FEATURES)
ground_truth[SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE] = 1

print("inputs ", tuple(x_train.shape), " targets", tuple(y_train.shape))
print(f"ground truth: feature {SIGNAL_FEATURE}, steps {SIGNAL_START}-{SIGNAL_END - 1}")

## 2. Train a small GRU

About 20 seconds on CPU.

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(N_FEATURES, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out.mean(dim=1))


model = GRUForecaster()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(25):
    perm = torch.randperm(len(x_train))
    for i in range(0, len(x_train), 128):
        idx = perm[i : i + 128]
        opt.zero_grad()
        loss_fn(model(x_train[idx]), y_train[idx]).backward()
        opt.step()

model.eval()
with torch.no_grad():
    r2 = 1 - loss_fn(model(x_test), y_test).item() / y_test.var().item()
print(f"test R2 = {r2:.4f}   (needs to be high, or there is no signal to explain)")

## 3. Explain it — this is the whole API

Pass the model, pass the inputs. `WinTSR` wraps any callable that maps
`(batch, seq_len, n_features)` to predictions.

In [ ]:
from tslens import WinTSR

inputs = x_test[:16]
baselines = torch.zeros_like(inputs)

attr = WinTSR(model).attribute(inputs, baselines=baselines, threshold=0.5)

print("attributions:", tuple(attr.shape), " (batch, n_output, seq_len, n_features)")

## 4. Look at it

The heatmap should be bright exactly on feature 0, steps 20–25.

In [ ]:
import matplotlib.pyplot as plt

saliency = attr.abs().mean(dim=1).detach()  # average over output steps

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

axes[0].imshow(ground_truth.T, aspect="auto", cmap="Greys", vmin=0, vmax=1)
axes[0].set_title("Ground truth")

for ax, i in zip(axes[1:], [0, 1]):
    ax.imshow(saliency[i].T, aspect="auto", cmap="viridis")
    ax.set_title(f"WinTSR — test sample {i}")

for ax in axes:
    ax.set_xlabel("time step")
    ax.set_ylabel("feature")
    ax.set_yticks(range(N_FEATURES))

plt.tight_layout()
plt.show()

## 5. Score it against the ground truth

Average precision of the saliency map against the known mask. Random guessing scores
about 0.024 here, since the signal covers 6 of 250 cells.

In [ ]:
from sklearn.metrics import average_precision_score


def average_precision(attr):
    a = attr.abs()
    if a.dim() == 4:
        a = a.mean(dim=1)
    rows = a.reshape(len(a), -1).detach().numpy()
    truth = ground_truth.reshape(-1).numpy()
    return float(np.mean([average_precision_score(truth, r) for r in rows]))


print(f"WinTSR average precision : {average_precision(attr):.4f}")
print(f"random baseline          : {ground_truth.mean():.4f}")

## 6. Compare against other attribution methods

WinTSR is Captum-compatible, so baselines are a one-liner each.

> **Note on what this toy task does and does not show.** It confirms every method,
> WinTSR included, recovers the planted window far above chance. It is *not* a
> benchmark: a single synthetic signal on a 5-channel AR series is much easier than the
> real forecasting and classification tasks in the paper, and the ranking here does not
> reproduce the paper's. For the actual evaluation — 5 datasets, 11 methods, and
> comprehensiveness / sufficiency metrics — see the `scripts/` folder in the repository
> and [arXiv:2412.04532](https://arxiv.org/abs/2412.04532).

In [ ]:
from captum.attr import IntegratedGradients
from tint.attr import FeatureAblation, Occlusion

methods = {
    "WinTSR": attr,
    "Occlusion": Occlusion(model).attribute(
        inputs, sliding_window_shapes=(1, 1), baselines=baselines, attributions_fn=abs
    ),
    "Feature Ablation": FeatureAblation(model).attribute(
        inputs, baselines=baselines, attributions_fn=abs
    ),
    "Integrated Gradients": IntegratedGradients(model).attribute(
        inputs, baselines=baselines
    ),
}

print(f"{'method':<24}{'avg precision':>14}")
print("-" * 38)
for name, a in sorted(methods.items(), key=lambda kv: -average_precision(kv[1])):
    print(f"{name:<24}{average_precision(a):>14.4f}")

## Next steps

- **Tune it.** `threshold` (0–1) controls how many low-relevance time steps are skipped —
  higher is faster and sparser. `sliding_window_shapes=(w, 1)` widens the temporal window.
- **Multi-input models.** Pass a tuple of tensors; you get a tuple of attributions back.
- **Reproduce the paper.** `WinTSR(model, legacy_normalize=True)` restores the exact
  normalization used to produce the published numbers.

If this was useful, please ⭐ the
[repository](https://github.com/khairulislam/tslens) and cite the paper.

```bibtex
@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```